In [1]:
import torch

In [2]:
torch.nn.TransformerEncoderLayer(d_model=128, nhead=4)

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (linear1): Linear(in_features=128, out_features=2048, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=2048, out_features=128, bias=True)
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
)

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

In [ ]:
import pandas as pd
import random
from tqdm import tqdm
from os.path import join as pjoin
from sklearn.metrics import r2_score, mean_absolute_error

import os
import copy
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from os.path import join as pjoin

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from torch import nn
from torch import optim

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from utils.misc import load_config, load_model_from_checkpoint, precision_recall_f1_score
from datasets.data_preparation import prepare_data

In [ ]:
root = '../../data/processed'

exp_path = '../results/dl.vit/2023-09-08 18Hr 45Min 49Sec IST+0530'

config = load_config('..', exp_path, 'hyperparameters.yaml')

In [ ]:
trainer_config = config['trainer']
data_config = config['data']

model_name = config['model']['__name__']

checkpoint_name = trainer_config['checkpoint_name']
device = trainer_config['device']
batch_size = trainer_config['batch_size']

patch_size = data_config['patch']['patch_size']
lithology_classes = data_config['lithology_classes']

config['root'] = '../..'

In [ ]:
from model.vit_autoregressor import build_model
from engine.autoregressor import validation_engine as blind_engine

model = build_model(config)

In [ ]:
x_train, x_val, y_train, y_val, num_classes = prepare_data(config, scaler_save = False)

In [ ]:
lithology_names = {v: k for k, v in lithology_classes.items()}

In [ ]:
train_dataset = TensorDataset(x_train, 
                                y_train)
train_loader = DataLoader(train_dataset, 
                            batch_size=trainer_config['batch_size'], 
                            shuffle=True)

val_dataset = TensorDataset(x_val, 
                            y_val)
val_loader = DataLoader(val_dataset, 
                        batch_size=trainer_config['batch_size'], 
                        shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

# Define the loss function and optimizer
regression_criterion = torch.nn.MSELoss()
classification_criterion = nn.CrossEntropyLoss(weight=torch.tensor(data_config['class_weights']).float().to(device))
loss_weights = config['model']['loss_weights']

In [ ]:
num_epochs = trainer_config['epochs']

In [ ]:
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=trainer_config['lr'])

In [ ]:
from engine.autoregressor import train_engine
from einops import rearrange

In [ ]:
for epoch in range(num_epochs):
    train_engine(
            epoch, 
            model, 
            train_loader, 
            regression_criterion,
            classification_criterion, 
            optimizer, 
            num_epochs, 
            loss_weights,
            device,
            patch_size,
            num_classes
        )
    
    val_correct = 0
    gt_val, pred_val = [], []
    model.eval()
    for batch_inputs_val, batch_labels_val in tqdm(val_loader, 
                                                    total=len(val_loader), 
                                                    desc=f"Val - Epoch {epoch+1}/{num_epochs}"):

        batch_inputs_val = batch_inputs_val.to(device)
        batch_labels_val = batch_labels_val.to(device)
        # autoregressive outputs initialized with zeros
        val_outputs = torch.zeros((batch_inputs_val.shape[0], patch_size, num_classes+2)).to(device)
        val_vit_embedding = None

        with torch.no_grad():
            for step in range(patch_size):
                lith_output_val, phi_output_val, sw_output_val, val_vit_embedding = model(batch_inputs_val, val_outputs, step, val_vit_embedding)
                val_next_token = torch.cat([lith_output_val, phi_output_val, sw_output_val], axis = -1)
                val_updated_initilizer = val_outputs.clone()
                val_updated_initilizer[:, step, :] = val_next_token
                val_outputs = val_updated_initilizer

        lith_batch_labels_val = batch_labels_val[:, :, 0]

        lith_output = val_outputs[:, :, :num_classes]

        lith_batch_labels_val = lith_batch_labels_val.long()

        outputs_ = rearrange(lith_output, 'b n d -> b d n')
        lith_loss_val = classification_criterion(outputs_, lith_batch_labels_val)

        val_predicted = torch.argmax(nn.Softmax(dim = -1)(lith_output), dim=-1)
        val_correct += (((val_predicted == lith_batch_labels_val).sum(-1).float().mean().item())/batch_inputs_val.shape[1])*100

        gt_val.append(lith_batch_labels_val.cpu())
        pred_val.append(val_predicted.cpu())

    cm_val = confusion_matrix(torch.cat(gt_val, dim=0).view(-1), torch.cat(pred_val, dim=0).view(-1))
    val_accuracy = val_correct / len(val_loader)
    break

In [ ]:
(((val_predicted == lith_batch_labels_val).sum(-1).float().mean().item())/batch_inputs_val.shape[1])*100

In [ ]:
(val_predicted == lith_batch_labels_val).sum().item()/(lith_batch_labels_val.shape[0]*lith_batch_labels_val.shape[1])*100

In [ ]:
val_outputs.shape

In [ ]:
val_accuracy